PROGETTO 4 - RICONSCIMENTOO FACCIALE IN AMBIENTI REALLI

Il riconoscimento facciale reale non servono solo modelli accurati ma sistemi resilienti, capaci di gestire imprevisti e rispondere in brevi tempi, anche quando la camera non è in posizione perfetta.

Ambiente imprevebile
Superare i limiti dei dataset controllati per applicazioni outdoor e indoor variate.
In un ambiente controllato, il riconoscimento facciale raggiunge accuratezze prossime alla perfezione. Tuttavia, nel deploy reale ci scontriamo con il rumore ambientale, ombre portate e rotazioni dell'asse del volto che degradano drasticamente gli embedding. 

Tecniche di pre-processing e l'allineamento geometrico possono compensare queste variazione prima che l'immagine raggiunga la rete neurale, garantendo una 'standarizzazione' dell'input necessaria per la stabilità del sistema.

Come possiamo rendere il nostro input digeribile per la rete?

Concetti Chiave per la Robustessa
- Normalizzazione della luce come un bilanciamento del bianco, utilizzo di algoritmo come CLAHE per contrastare ombre eccessive o sovraesposizione locale.
- Face Alignment è come raddirazzare una foto storata affinche gli occhi siano sempre sulla stessa linea
- Data Augmentation Sinttica addestramento su variazione di gamma e rotazione per rendere il modello invariante a tali parametri
- Frontalizzazione: tecniche di proiezione 3D per ricostruire una vista frontale partendo da un volto di profilo.

Strumenti di Pre-processing
- CLAHE e Correzione Gamma: Il Contrast Limited Adaptive Histogram Equilization permette di bilanciare il contrasto in modo locale, evitando che zone d'omnra nascondano feature biometriche essenziali per il calcolo dei descrittori. Il CLAHE illumina solo gli angoli scuri senza modificare le zone chiare.
- Landmark-based Alignment: individuare la posizione degli occhi e della punta del naso permette di ruotare l'immagine in modo che la linea interoculare si asempre orizzontale, minimizzando la varianza che il modello deve gestire.
- Filtraggio del Rumore: l'applicazione di filtri bilaterali o gaussiani può ridurre il rumore del sensore in condizioni di scarsa luminosità senza però smussare eccessivamente i bordi dei tratti somatici.

Una volta riconosciuto ed indivisuato un volto, dobbiamo ricordarci chi è

Sistema di Logging degli Accessi
Tracciare e storicizzare gli eventi di riconoscimento per la sicurezza e l'analisi
Un sistema di riconoscimento facciale non è completo senza una logica di persistenza. Dobbiamo essere in grado di registrare chi è stato visto, in quale momento e con quale gradi di confidenza.
Dobbiamo progettare un'interfaccia di log che separi la logica di inferenza da quella di scrittura su disco, assicurando che le operazioni di I/O non rallentino il thread principale di visione.

Architettura del Logger
Componenti per la gestione dei dati storici
- Event-Driven Logging: attivazione della scrittura solo quando viene superata una soglia di confidenza specifica per un volto nuovo.
- De-duplicazione Temporale: evitare di loggare lo stesso utente 30 volte al secondo implementando finestre temporali di cooldown
- Storage Strutturato: memorizzare in database SQLlite o file CSV per una rapida consultazione e analisi post-evento
- Integrazione Media: salvataggio del frame (ritaglio del volto catturato in quel momento) insieme al timestamp per una verifica visiva umana a posteriori.

Ma dove mettiamo tutti questi dati?
la sfida database o filesystem?

Gestione dei DAti di Accesso
Database vs File System
Mentre i metadati (nomi, orari, confidenza) risiedono meglio in un DB relazionale, i ritagli delle immagini devono essere gestiti nel file system con percorsi indicizzati nel DB per scalabilità
Privacy e Data Retention
Invece di iterare su una lista Python per confrontare gli embedding utilizzeremo operazione matriciali NumPy o librerie come FAISS per ricerche sub-millisecondo su grandi database
Asincronicità dei Log
L'utilizzo di code (Queue) permette di delegare la scrittura del log a un processo secondrio, mantenendo fluida la visualizzazione del feed video a 30 FPS

Come fa il sistema se decidere se loggare o meno
Soglie di Riconoscimento e Confidenza
Metriche di decisione per il log
La decisione di loggare un accesso dipende dalla distanza euclidea tra l'embedding rilevato e quelli presenti nel database. Minore è la distanza, maggiore è la probabilità di matching corretto.
Definiamo la distanza critica come il limite oltre il quale l'identità viene considerata sconosciuta, minimizzando i falsi positivi.

Ora che il sistema sa ricordare deve diventare veloce

Ottimizzazione delle Performance
La latenza è il nemico numero  uno dell'esperienza utente. Un ritardo di mezzo secondo tra il passaggio di una persona e il riconoscimento può rendere il sistema inutilizzabile in varchi ad alto flusso.
Si possono ottimizzare l'implementazione Python sfruttando la riduzione della risoluzione operativa e ilmultithreading per ottenere una rispostas in tempo reale.

Tecniche di Accelerazione
Strategie per ridurre il carico computazionale
- Frame Skipping: processare l'ingerenza solo ogni N frame, mentre per i restanti si utilizza un tracker leggere per seguire il volto
- Downscaling dell'input: ridurre la risoluzione del frame per la fase di rilevamento, mantenendo l'alta qualità solo per l'estrazione degli embedding
- Inference Batching: raggruppare più volti rilevati nello stesso frame per processarli con una singola chiamata alla GPE/NPU
- Modelli Quantizzati: utilizzo di pesi in formato FP16 o INT8 per accelerare il calcolo tensoriale senza perdita significativa di accuratezza.

Implementazione Efficiente.
In Python, separare il thread di acquisizine della camera (OpenCV) dal thread di inferenza Deep Learning evita il buffering dei frame obsoleti.
Invece di iterare su una lista Python per confrontare gli embedding, utilizzeremo operazioni matriciali Numpy o librerie FAISS per ricerche sub-millisecondo su grandi database
Identifare i colli di bottiglia tramite toolcome 'cProfile' o 'line_profiler' per capire se il tempo è speso nell'I/O, nel pre-processing o nell'inferenza pura.

In [1]:
"""
================================================================================
AI-GUARD PRO SOTA 2026: SISTEMA DI SORVEGLIANZA INTELLIGENTE
================================================================================
DESCRIZIONE DEL FLUSSO OPERATIVO:
1. ACQUISIZIONE: Il sistema legge i frame dalla webcam in tempo reale.
2. DETECTION & TRACKING (YOLO): Ogni singolo frame viene analizzato per trovare persone.
   YOLO assegna un ID unico a ogni persona che "segue" nel tempo (Tracking).
3. BIOMETRIA SELETTIVA: Per non rallentare il sistema, il riconoscimento facciale
   NON avviene su ogni frame, ma solo quando appare un nuovo ID o ogni N frame.
4. AZIONI ASINCRONE: Se viene rilevato un estraneo, il sistema salva una foto su disco
   usando un thread separato, così il video non scatta (Lag-free).
5. LOGGING: Ogni evento viene registrato in un file CSV per scopi di audit.

INTERAZIONI CHIAVE:
- SecurityEngine <-> YOLO: Il motore interroga il modello neurale per le coordinate (box).
- SecurityEngine <-> face_recognition: Confronta i volti rilevati con il database.
- Processi paralleli: Il thread principale gestisce il video, thread secondari i salvataggi.
================================================================================
"""

import os
import cv2
import numpy as np
import time
import csv
import threading
from datetime import datetime

# --- CONFIGURAZIONE BACKEND AI ---
# Integriamo Keras 3 configurandolo con PyTorch per massime prestazioni su GPU.
os.environ["KERAS_BACKEND"] = "torch"
import torch
from ultralytics import YOLO
import face_recognition

class SecurityEngine:
    """
    CLASSE CORE: Coordina l'intelligenza artificiale e la gestione dei dati.
    Agisce come 'cervello' del sistema collegando sensore (webcam) e database.
    """
    def __init__(self, known_encodings, known_names):
        """
        Inizializza i modelli e prepara l'ambiente di archiviazione.
        :param known_encodings: Lista di 'impronte digitali' dei volti noti.
        :param known_names: Lista di nomi corrispondenti alle impronte.
        """
        # Scegliamo se usare la scheda video (CUDA) o il processore (CPU)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"[*] Inizializzazione AI su: {self.device.upper()}")

        # Carichiamo YOLO v11 (SOTA 2026) e lo spostiamo sulla GPU per velocità estrema
        self.detector = YOLO("yolo11n.pt").to(self.device)
        
        # Database interno dei volti autorizzati
        self.known_encodings = known_encodings
        self.known_names = known_names
        
        # Dizionario per "ricordare" chi è chi durante la sessione {ID_YOLO: Nome}
        self.tracked_identities = {} 
        
        # Gestione cartelle per log e screenshot degli intrusi
        self.logs_dir = "security_data"
        self.intruders_dir = os.path.join(self.logs_dir, "intruders")
        os.makedirs(self.intruders_dir, exist_ok=True) # Crea le cartelle se mancano
        
        self.log_file = os.path.join(self.logs_dir, "access_log.csv")
        self._init_csv() # Prepara il file CSV
        
        # Timestamp per evitare di intasare il log (scrive un evento ogni 30 sec per persona)
        self.last_log_time = {} 

    def _init_csv(self):
        """Crea il file Excel/CSV di log con l'intestazione se non esiste."""
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                csv.writer(f).writerow(["Data", "Ora", "Identita", "Stato"])

    def _async_save_intruder(self, frame, track_id):
        """
        LOGICA ASINCRONA: Salva l'immagine dell'intruso in background.
        Perché? Scrivere un file su Hard Disk è lento. Facendolo qui, il video resta fluido.
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"intruder_id{track_id}_{timestamp}.jpg"
        filepath = os.path.join(self.intruders_dir, filename)
        
        # Aggiungiamo una scritta rossa "ALLERTA" direttamente sulla foto salvata
        cv2.putText(frame, f"ALLERTA INTRUSO: {timestamp}", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
        cv2.imwrite(filepath, frame) # Scrittura fisica del file
        print(f"[!] PROVA ACQUISITA E SALVATA: {filename}")

    def log_and_capture(self, name, track_id, full_frame):
        """
        Registra l'evento nel database e attiva la cattura foto se necessario.
        Interagisce con il file system e il sistema di threading.
        """
        now = time.time()
        # Cooldown: non loggare la stessa persona più di una volta ogni 30 secondi
        if name not in self.last_log_time or (now - self.last_log_time[name]) > 30:
            status = "AUTORIZZATO" if name != "ESTRANEO" else "NON AUTORIZZATO"
            
            # Scrittura riga nel file CSV
            with open(self.log_file, 'a', newline='') as f:
                d = datetime.now()
                csv.writer(f).writerow([d.strftime("%Y-%m-%d"), d.strftime("%H:%M:%S"), name, status])
            
            # Se la IA non riconosce la persona, lancia il thread di salvataggio foto
            if name == "ESTRANEO":
                threading.Thread(target=self._async_save_intruder, 
                                 args=(full_frame.copy(), track_id)).start()
            
            self.last_log_time[name] = now
            print(f">>> [LOG] Evento registrato per: {name} ({status})")

    def apply_robustness(self, crop):
        """
        SOTA PRE-PROCESSING: Migliora la qualità dell'immagine prima del riconoscimento.
        Usa CLAHE per bilanciare luci ed ombre, rendendo il volto chiaro anche in controluce.
        """
        if crop.size == 0: return crop
        # Convertiamo in spazio colore LAB (Luce, A-canale, B-canale)
        lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        # Applichiamo l'equalizzazione adattiva solo al canale della luminosità (L)
        l = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)
        # Ricomponiamo l'immagine migliorata
        crop = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)
        return crop

    def start_surveillance(self):
        """
        PIPELINE PRINCIPALE: Il cuore pulsante del sistema.
        Gestisce il ciclo infinito di visione, calcolo e visualizzazione.
        """
        cap = cv2.VideoCapture(0) # Apriamo la webcam predefinita
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280) # Alta definizione
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        
        frame_count = 0 # Contatore per la biometria temporizzata
        
        print("\n--- MONITORAGGIO AI-GUARD PRO ATTIVO (Premi 'q' per uscire) ---")

        while cap.isOpened():
            ret, frame = cap.read() # Leggiamo un fotogramma
            if not ret: break
            frame_count += 1

            # --- FASE 1: TRACKING (Veloce, avviene ogni frame) ---
            # YOLO individua le persone (class=0) e assegna loro un "track_id" che le segue
            results = self.detector.track(frame, persist=True, classes=[0], verbose=False, device=self.device)

            # Se YOLO ha trovato qualcuno e ha assegnato degli ID di tracciamento
            if results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.int().cpu().numpy() # Coordinate del rettangolo
                ids = results[0].boxes.id.int().cpu().numpy()    # ID numerico della persona

                for box, track_id in zip(boxes, ids):
                    x1, y1, x2, y2 = box # Estraiamo i bordi del rettangolo
                    
                    # --- FASE 2: RICONOSCIMENTO (Campionato, avviene ogni 15 frame) ---
                    # Ottimizzazione: non facciamo biometria ogni millisecondo (pesante), 
                    # ma solo se l'ID è nuovo o è passato circa mezzo secondo (15 frame).
                    if track_id not in self.tracked_identities or frame_count % 15 == 0:
                        crop = frame[y1:y2, x1:x2] # Ritagliamo l'area della persona
                        face_ready = self.apply_robustness(crop) # Pulizia immagine (CLAHE)
                        rgb_crop = cv2.cvtColor(face_ready, cv2.COLOR_BGR2RGB) # Formato per dlib/face_rec
                        
                        # Cerchiamo i volti nel ritaglio
                        locs = face_recognition.face_locations(rgb_crop)
                        if locs:
                            # Trasformiamo il volto in un vettore matematico (encoding)
                            encs = face_recognition.face_encodings(rgb_crop, locs)
                            if encs:
                                # Confrontiamo con il nostro database di persone note
                                matches = face_recognition.compare_faces(self.known_encodings, encs[0], tolerance=0.5)
                                # Se c'è un match prendiamo il nome, altrimenti è un "ESTRANEO"
                                name = self.known_names[matches.index(True)] if True in matches else "ESTRANEO"
                                
                                # Salviamo il risultato nella memoria a breve termine del sistema
                                self.tracked_identities[track_id] = name
                                # Chiamiamo il sistema di logging e sicurezza
                                self.log_and_capture(name, track_id, frame)

                    # --- FASE 3: RENDERING UI (Visualizzazione grafica) ---
                    # Recuperiamo l'identità associata a quel track_id
                    identity = self.tracked_identities.get(track_id, "In analisi...")
                    
                    # Colore: Verde se autorizzato, Rosso se estraneo, Giallo se sta ancora calcolando
                    color = (0, 255, 0) if identity not in ["ESTRANEO", "In analisi..."] else (0, 0, 255)
                    if identity == "In analisi...": color = (0, 255, 255)

                    # Disegniamo il rettangolo e il testo sul video
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(frame, f"ID:{track_id} {identity}", (x1, y1 - 10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # Mostriamo il risultato finale a video
            cv2.imshow("AI-GUARD PRO SOTA 2026", frame)
            
            # Esci se premi il tasto 'q'
            if cv2.waitKey(1) & 0xFF == ord('q'): break
            
        # Pulizia finale delle risorse
        cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    # --- SETUP DATABASE BIOMETRICO ---
    # In una situazione reale, qui caricheresti le foto dei dipendenti o familiari.
    # Esempio: 
    #   per_enc = face_recognition.face_encodings(face_recognition.load_image_file("persona.jpg"))[0]
    #   encodings_database = [per_enc]
    #   names_database = ["Mario Rossi"]
    
    encodings_database = []
    names_database = []
    
    # Avvio dell'applicazione
    app = SecurityEngine(encodings_database, names_database)
    app.start_surveillance()

[*] Inizializzazione AI su: CPU

--- MONITORAGGIO AI-GUARD PRO ATTIVO (Premi 'q' per uscire) ---
